# 3 · MIF / MIF-ST - sequence design
`sequence-models` package, autoregressive design in random order.
**Cleaned for comparability:** uses the shared v6 structures (not v4 re-downloads)
and temperature-only sampling (top-k/top-p disabled) to match the other models.
**Runtime → GPU.**

In [ ]:
#@title Step 0 - Upload & unzip the design bundle
#@markdown Upload **design_bundle.zip** (contains `design_common.py`,
#@markdown `design_input_proteins.csv`, and `structures/`).
#@markdown Build it locally with `design/make_bundle.sh`.
import os, zipfile
from google.colab import files

if not os.path.exists("design_common.py"):
    print("Upload design_bundle.zip:")
    up = files.upload()
    zname = next(iter(up))
    with zipfile.ZipFile(zname) as z:
        z.extractall(".")
    # if it unzipped into a 'design/' subdir, hoist contents to CWD
    if os.path.exists("design/design_common.py") and not os.path.exists("design_common.py"):
        import shutil
        for item in os.listdir("design"):
            shutil.move(os.path.join("design", item), item)
print("Bundle ready:", sorted(os.listdir(".")))

In [ ]:
#@title Install sequence-models (MIF)
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","--quiet",
                "sequence-models","tqdm"], check=True)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (set GPU runtime!)")

In [ ]:
#@title Step 1 - Import shared config and show the LOCKED settings
import design_common as dc
proteins = dc.load_inputs()           # 25 templates, structure paths resolved
print(f"Loaded {len(proteins)} design templates")
print("\n=== LOCKED CONFIG (identical across all model notebooks) ===")
import dataclasses, json
cfg = {k: v for k, v in dataclasses.asdict(dc.CONFIG).items() if k != "deviations"}
print(json.dumps(cfg, indent=2, default=str))
display(proteins[["uniprot_id","species","domain","rank_class","sequence_length"]])

In [ ]:
#@title Choose variant
variant = "mif"  #@param ["mif", "mifst"]
MODEL = {"mif":"MIF","mifst":"MIF-ST"}[variant]
SOLUBLE = False
import torch
from sequence_models.pretrained import load_model_and_alphabet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mif_model, collater = load_model_and_alphabet(variant)
mif_model = mif_model.eval().to(device)
# CRITICAL: MIF uses PROTEIN_ALPHABET (30 tokens) with MASK='#' (id 28), NOT trR_ALPHABET.
# Derive the true alphabet + mask token from the collater's tokenizer (same source the
# scoring notebook uses). Masking with the wrong token / indexing the wrong alphabet
# produces near-random designs (~12% recovery) - verified failure mode.
_tok = collater.sequence_collater.tokenizer
ALPHABET  = _tok.alphabet                 # 'ACDEFGHIKLMNPQRSTVWYBZXJOU*-#@'
MASK_TOK  = ALPHABET[_tok.mask_id]         # '#'
CANON_IDX = [ALPHABET.index(a) for a in dc.CANONICAL_AA]
print("Loaded", MODEL, "on", device, "| mask token:", repr(MASK_TOK), "| alphabet len:", len(ALPHABET))

## Comparability notes - MIF / MIF-ST

These are the points where MIF / MIF-ST touches the locked settings. Anything that
**deviates** is recorded via `dc.CONFIG.note_deviation(...)` so it lands in the
output manifest.


- **Alphabet/mask (critical fix)**: MIF uses `PROTEIN_ALPHABET` with mask token `'#'`
  (id 28). The original notebook used `trR_ALPHABET` and masked with `'-'` (PAD),
  which made the model emit near-random sequences (~12% WT recovery). Now derived
  from the collater's tokenizer, matching the scoring notebook. Verified: recovery
  jumps to ~55%.
- **Structure**: reads the shared **v6** PDB from the bundle (original re-downloaded v4).
- **Sampler**: original used `top_k=8, top_p=0.95`; **disabled** to match the locked
  temperature-only scheme.
- **Decoding**: autoregressive in a random position order (faithful to MIF's design mode).
- **score_type** = `mean_logp` (mean log-prob of chosen residues over the trajectory).

In [ ]:
#@title MIF design helpers (structure prep + temperature-only autoregressive sampling)
import numpy as np, torch
from sequence_models.pdb_utils import parse_PDB, process_coords

def prep_struct(pdb_path):
    coords, wt_seq, _ = parse_PDB(pdb_path)
    cds = {"N":coords[:,0], "CA":coords[:,1], "C":coords[:,2]}
    dist, omega, theta, phi = process_coords(cds)
    t = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
    return wt_seq, (t(dist), t(omega), t(theta), t(phi))

@torch.no_grad()
def design_one(wt_seq, struct, seed):
    """Autoregressive, temperature-only (no top-k/p) to match the locked scheme."""
    rng = np.random.default_rng(seed)
    L = len(wt_seq)
    seq = list(MASK_TOK * L)          # start all-masked with the TRUE mask token '#'
    order = rng.permutation(L)
    dist, omega, theta, phi = struct
    logps = []
    for pos in order:
        batch = [["".join(seq), dist, omega, theta, phi]]
        src, nodes, edges, conn, emask = collater(batch)
        src,nodes,edges,conn,emask = [x.to(device) for x in (src,nodes,edges,conn,emask)]
        logits = mif_model(src, nodes, edges, conn, emask, result="logits")[0, pos]
        # restrict to the 20 canonical AAs (indices into the model's real alphabet)
        sub = logits[CANON_IDX] / dc.CONFIG.temperature
        probs = torch.softmax(sub, dim=-1)
        choice = torch.multinomial(probs, 1).item()
        seq[pos] = dc.CANONICAL_AA[choice]
        logps.append(float(torch.log(probs[choice] + 1e-12)))
    return "".join(seq), float(np.mean(logps))

In [ ]:
#@title Step 2 - Smoke test (shortest protein, one seed)
_p = proteins.sort_values("sequence_length").iloc[0]
_wt, _st = prep_struct(_p.structure_path)
_seq, _lp = design_one(_wt, _st, seed=dc.CONFIG.seeds[0])
print(f"{_p.uniprot_id} len={_p.sequence_length}  mean_logp={_lp:.3f}")
print("DES:", _seq[:60])
assert len(_seq) == _p.sequence_length and set(_seq) <= set(dc.CANONICAL_AA)
print("✓ smoke test OK  (note: autoregressive - full run is slower)")

In [ ]:
#@title Step 3 - Design all 25 proteins
from tqdm.auto import tqdm
rows = []
for p in tqdm(list(proteins.itertuples()), desc=f"{MODEL} design"):
    wt, st = prep_struct(p.structure_path)
    for i, seed in enumerate(dc.CONFIG.seeds):
        seq, lp = design_one(wt, st, seed=seed)
        rows.append(dc.make_record(p, model=MODEL, sample_idx=i, seed=seed,
                                   designed_sequence=seq, model_score=lp,
                                   score_type="mean_logp", soluble_variant=SOLUBLE))
print(f"Generated {len(rows)} sequences")

In [ ]:
#@title Step 4 - Validate (faithful + comparable) and write outputs
df = dc.finalize(rows, model=MODEL, strict=True)   # raises if a guard fails
dc.write_designs(df, MODEL)
dc.write_fasta(df, MODEL)

# Quick faithfulness readout: per-protein WT sequence recovery distribution
import numpy as np
rec = df.apply(lambda r: sum(a==b for a,b in zip(r.designed_sequence, r.wt_sequence))/r.seq_length, axis=1)
print(f"\nSeq-recovery vs WT - median {rec.median():.1%}, "
      f"IQR [{rec.quantile(.25):.1%}, {rec.quantile(.75):.1%}]")
print("(Inverse-folding designs typically recover ~30-55% of WT; "
      "near-100% means the sampler is too cold / stuck, near-5% means random.)")

from google.colab import files
files.download(str(dc.OUTPUT_DIR / f"designs_{MODEL.replace('/','_').replace('-','-')}.csv"))